# CS-781 Phase 1 — Architecture 1: decoder-only LegalLM from scratch (new BPE tokenizer)

In this notebook I train my decoder-only legal language model from scratch on the official Phase-1 corpus and generate the 128-word continuations for the test prompts.

**Model.** 12 layers, d_model 768, 12 heads, RoPE positions, RMSNorm, SwiGLU FFN (d_ff 2048), context 1024, tied input/output embeddings, 109.5M parameters (the 50M–300M rule is asserted in code).

**Tokenizer.** My from-scratch byte-level BPE (`a1_tokenization.py` + `vocab.json` / `merges.txt`, 32k vocabulary).

**Data.** The four official training files are streamed from Hugging Face and tokenised into shards with a source-weighted mix (Supreme Court 0.40, High Courts 0.25, Tribunals 0.25, Acts 0.10); a held-out split per source is kept for validation.

**Pipeline.**
1. Fail-early checks: model unit test, tiny data, tiny training run.
2. Data preparation (resumable).
3. Training on 2 GPUs with DDP: AdamW (0.9, 0.95), warmup + cosine decay fitted to the time budget, fp16, gradient clipping 1.0, checkpoints during training.
4. Decoding sweep on held-out prompts, chosen by BERTScore (greedy, repetition penalty, n-gram blocking, top-p), then generation of the 1000 test continuations (exactly 128 words each) and `submission.zip`.


In [1]:
BUDGET_HOURS = 7.5      # training time. 6-8.5 is safe inside one 12 h session
MIX = "supreme_court:0.40,high_courts:0.25,tribunals:0.25,acts:0.10"   # data-mix experiment = change this
import os, glob, torch
from kaggle_secrets import UserSecretsClient
for k in ("HF", "HF_TOKEN"):
    try: os.environ["HF_TOKEN"] = UserSecretsClient().get_secret(k); break
    except Exception: pass
assert os.environ.get("HF_TOKEN"), "add the Kaggle secret named HF"
assert glob.glob("/kaggle/input/**/a1_tokenization.py", recursive=True), "add the tokenizer dataset as input"
assert glob.glob("/kaggle/input/**/vocab.json", recursive=True) and glob.glob("/kaggle/input/**/merges.txt", recursive=True)
assert torch.cuda.device_count() == 2, "set accelerator to GPU T4 x2"
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(2)], "| budget", BUDGET_HOURS, "h")

GPUs: ['Tesla T4', 'Tesla T4'] | budget 7.5 h


In [2]:
!pip -q install ijson requests bert-score rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00


In [3]:
%%writefile /kaggle/working/legal_lm.py
"""legal_lm.py - decoder-only language model written from scratch in plain PyTorch.

RoPE positions, RMSNorm (pre-norm), SwiGLU MLP, F.scaled_dot_product_attention (causal),
tied embeddings, no biases. Includes a batched KV-cache generator (left-padded batches).
Separate q/k/v/o and gate/up/down Linear layers so LoRA/LoRMA/adapters attach easily later.
"""
import json
import math
import os
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int = 32000
    n_layer: int = 12
    n_head: int = 12
    d_model: int = 768
    d_ff: int = 2048
    max_seq_len: int = 1024
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    dropout: float = 0.0
    tie_embeddings: bool = True
    pad_id: int = 0
    bos_id: int = 1
    eos_id: int = 2


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        y = x.float()
        y = y * torch.rsqrt(y.pow(2).mean(-1, keepdim=True) + self.eps)
        return y.to(x.dtype) * self.weight


def rope_cos_sin(pos_ids, head_dim, theta):
    """pos_ids (B,T) long -> cos,sin of shape (B,1,T,head_dim/2), float32 (computed fresh: safe for .half())."""
    inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=pos_ids.device, dtype=torch.float32) / head_dim))
    f = pos_ids.float().unsqueeze(-1) * inv
    return f.cos().unsqueeze(1), f.sin().unsqueeze(1)


def apply_rope(x, cos, sin):
    dt = x.dtype
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat((x1 * cos - x2 * sin, x2 * cos + x1 * sin), dim=-1).to(dt)


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg.d_model
        self.h, self.hd, self.p = cfg.n_head, d // cfg.n_head, cfg.dropout
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, d, bias=False)
        self.v_proj = nn.Linear(d, d, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)

    def forward(self, x, cos, sin, mask=None, cache=None, cache_len=0):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is None:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                               dropout_p=self.p if self.training else 0.0)
        else:
            ck, cv = cache
            ck[:, :, cache_len:cache_len + T] = k
            cv[:, :, cache_len:cache_len + T] = v
            n = cache_len + T
            y = F.scaled_dot_product_attention(q, ck[:, :, :n], cv[:, :, :n], attn_mask=mask)
        return self.o_proj(y.transpose(1, 2).reshape(B, T, -1))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = Attention(cfg)
        self.mlp_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin, mask=None, cache=None, cache_len=0):
        x = x + self.drop(self.attn(self.attn_norm(x), cos, sin, mask, cache, cache_len))
        return x + self.drop(self.mlp(self.mlp_norm(x)))


class LegalLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight
        self.drop = nn.Dropout(cfg.dropout)
        self.apply(self._init)
        std = 0.02 / math.sqrt(2 * cfg.n_layer)          # scaled init of residual projections
        for n, p in self.named_parameters():
            if n.endswith("o_proj.weight") or n.endswith("down_proj.weight"):
                nn.init.normal_(p, 0.0, std)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())     # tied weights counted once

    # ---- training / feature extraction ----
    def hidden_states(self, input_ids, pos_ids=None):
        """Final (post-norm) hidden states (B,T,d): attach classification / retrieval heads here in later phases."""
        B, T = input_ids.shape
        if pos_ids is None:
            pos_ids = torch.arange(T, device=input_ids.device)[None]
        cos, sin = rope_cos_sin(pos_ids, self.cfg.d_model // self.cfg.n_head, self.cfg.rope_theta)
        x = self.drop(self.tok_emb(input_ids))
        for blk in self.blocks:
            x = blk(x, cos, sin)
        return self.norm(x)

    def forward(self, input_ids, targets=None):
        logits = self.lm_head(self.hidden_states(input_ids))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    # ---- inference with KV cache ----
    def init_cache(self, B, max_len, device, dtype):
        h, hd = self.cfg.n_head, self.cfg.d_model // self.cfg.n_head
        return [(torch.zeros(B, h, max_len, hd, device=device, dtype=dtype),
                 torch.zeros(B, h, max_len, hd, device=device, dtype=dtype)) for _ in self.blocks]

    @torch.no_grad()
    def forward_cached(self, input_ids, pos_ids, mask, caches, cache_len):
        cos, sin = rope_cos_sin(pos_ids, self.cfg.d_model // self.cfg.n_head, self.cfg.rope_theta)
        x = self.tok_emb(input_ids)
        for blk, c in zip(self.blocks, caches):
            x = blk(x, cos, sin, mask, c, cache_len)
        return self.lm_head(self.norm(x[:, -1]))              # logits of the last position only (B,V)

    # ---- save / load ----
    def save_pretrained(self, weights_path, half=False):
        d = os.path.dirname(weights_path) or "."
        os.makedirs(d, exist_ok=True)
        with open(os.path.join(d, "model_config.json"), "w") as f:
            json.dump(asdict(self.cfg), f, indent=1)
        sd = {k: (v.half() if half and v.is_floating_point() else v) for k, v in self.state_dict().items()}
        torch.save(sd, weights_path)


def load_legal_lm(weights_path, device="cpu", dtype=torch.float32, config_path=None):
    if config_path is None:
        for d in (os.path.dirname(weights_path), os.path.dirname(os.path.dirname(weights_path))):
            c = os.path.join(d, "model_config.json")
            if os.path.exists(c):
                config_path = c
                break
    with open(config_path) as f:
        cfg = LMConfig(**json.load(f))
    model = LegalLM(cfg)
    sd = torch.load(weights_path, map_location="cpu")
    if "model" in sd and isinstance(sd["model"], dict):     # full training checkpoint
        sd = sd["model"]
    model.load_state_dict(sd)
    return model.to(device=device, dtype=dtype).eval()


# ======================================================================================
# Generation
# ======================================================================================
def build_starts_word(tok):
    """starts_word[id]=1 if the token begins with a space (= starts a new whitespace-delimited word)."""
    t = torch.zeros(len(tok.id_to_token), dtype=torch.long)
    for i, s in enumerate(tok.id_to_token):
        if s.startswith("\u0120"):
            t[i] = 1
    return t


def _sample(lg, temperature, top_k, top_p, generator):
    if temperature <= 0:
        return lg.argmax(-1)
    lg = lg / temperature
    if top_k > 0:
        kth = torch.topk(lg, top_k, dim=-1).values[:, -1:]
        lg = lg.masked_fill(lg < kth, float("-inf"))
    if top_p < 1.0:
        sl, si = torch.sort(lg, descending=True, dim=-1)
        pr = sl.softmax(-1)
        sl = sl.masked_fill((pr.cumsum(-1) - pr) > top_p, float("-inf"))
        lg = torch.full_like(lg, float("-inf")).scatter(-1, si, sl)
    return torch.multinomial(lg.softmax(-1), 1, generator=generator).squeeze(1)


@torch.no_grad()
def generate_batch(model, prompts, max_new, starts_word, target_words=128, margin=6,
                   temperature=0.0, top_k=0, top_p=1.0, rep_penalty=1.0, no_repeat_ngram=0,
                   banned_ids=(0, 1, 2, 3), generator=None):
    """prompts: list of token-id lists. Never emits EOS/pad/bos/unk. Stops when every sequence has
    >= target_words+margin words (counted by space-prefixed tokens). Returns list of generated id lists."""
    p0 = next(model.parameters())
    dev, dtype = p0.device, p0.dtype
    B = len(prompts)
    lens = torch.tensor([len(p) for p in prompts])
    L = int(lens.max())
    ids = torch.zeros(B, L, dtype=torch.long)
    valid = torch.zeros(B, L, dtype=torch.bool)
    for i, p in enumerate(prompts):                          # LEFT padding
        ids[i, L - len(p):] = torch.tensor(p)
        valid[i, L - len(p):] = True
    ids, valid, lens = ids.to(dev), valid.to(dev), lens.to(dev)
    caches = model.init_cache(B, L + max_new, dev, dtype)
    pos = (valid.long().cumsum(1) - 1).clamp(min=0)          # real tokens start at position 0
    causal = torch.tril(torch.ones(L, L, dtype=torch.bool, device=dev))
    mask = (causal[None, None] & valid[:, None, None, :]) | torch.eye(L, dtype=torch.bool, device=dev)[None, None]
    logits = model.forward_cached(ids, pos, mask, caches, 0)
    key_valid = torch.zeros(B, L + max_new, dtype=torch.bool, device=dev)
    key_valid[:, :L] = valid
    gen = torch.zeros(B, max_new, dtype=torch.long, device=dev)
    wc = torch.zeros(B, dtype=torch.long, device=dev)
    sw = starts_word.to(dev)
    n = no_repeat_ngram
    assert n == 0 or n >= 2
    hist = [[] for _ in range(B)]
    seen = [dict() for _ in range(B)]
    T = 0
    for t in range(max_new):
        lg = logits.float()
        lg[:, list(banned_ids)] = float("-inf")
        if rep_penalty != 1.0 and t > 0:
            g = gen[:, :t]
            sc = lg.gather(1, g)
            lg.scatter_(1, g, torch.where(sc < 0, sc * rep_penalty, sc / rep_penalty))
        if n and t >= n - 1:
            rows, cols = [], []
            for b in range(B):
                for c in seen[b].get(tuple(hist[b][-(n - 1):]), ()):
                    rows.append(b)
                    cols.append(c)
            if rows:
                lg[rows, cols] = float("-inf")
        nxt = _sample(lg, temperature, top_k, top_p, generator)
        gen[:, t] = nxt
        wc += sw[nxt]
        T = t + 1
        if n:
            nl = nxt.tolist()
            for b in range(B):
                hist[b].append(nl[b])
                if len(hist[b]) >= n:
                    seen[b].setdefault(tuple(hist[b][-n:-1]), set()).add(nl[b])
        if bool((wc >= target_words + margin).all()) or t == max_new - 1:
            break
        key_valid[:, L + t] = True
        logits = model.forward_cached(nxt[:, None], (lens + t)[:, None],
                                      key_valid[:, None, None, :L + t + 1], caches, L + t)
    return gen[:, :T].tolist()


def generate_texts(model, tok, prompts, gen_cfg=None, batch_size=48, max_prompt=768, max_new=256,
                   target_words=128, seed=0, verbose=True):
    """Returns one string of EXACTLY target_words whitespace-separated words per prompt (prompt not included).
    The LAST max_prompt tokens of an over-long prompt are kept. Short outputs are extended (up to 3 rounds)."""
    import time
    gen_cfg = dict(gen_cfg or {})
    sw = build_starts_word(tok)
    n = len(prompts)
    ids = [tok.encode(p)[-max_prompt:] for p in prompts]
    gens = [[] for _ in range(n)]
    texts = [None] * n
    pending = sorted(range(n), key=lambda i: len(ids[i]))
    dev = next(model.parameters()).device
    g = None
    if gen_cfg.get("temperature", 0) > 0:
        g = torch.Generator(device=dev)
        g.manual_seed(seed)
    t0 = time.time()
    for rnd in range(3):
        nxt = []
        for s in range(0, len(pending), batch_size):
            chunk = pending[s:s + batch_size]
            ctx = [(ids[i] + gens[i])[-max_prompt:] for i in chunk]
            out = generate_batch(model, ctx, max_new, sw, target_words=target_words, generator=g, **gen_cfg)
            for i, o in zip(chunk, out):
                gens[i] += o
                words = tok.decode(gens[i]).split()
                if len(words) >= target_words:
                    texts[i] = " ".join(words[:target_words])
                else:
                    nxt.append(i)
            if verbose:
                print(f"  round {rnd} batch {s // batch_size + 1}/{math.ceil(len(pending) / batch_size)} "
                      f"({time.time() - t0:.0f}s)", end="\r")
        pending = nxt
        if not pending:
            break
    for i in pending:                                        # last-resort fallback (should never trigger)
        w = tok.decode(gens[i]).split()
        w += ["the"] * (target_words - len(w))
        texts[i] = " ".join(w[:target_words])
    if verbose:
        print(f"\n  generated {n} texts in {time.time() - t0:.0f}s; fallback used for {len(pending)}")
    return texts

Writing /kaggle/working/legal_lm.py


In [4]:
%%writefile /kaggle/working/train.py
"""train.py - DDP pre-training of LegalLM (decoder-only) on 2x T4, fp16 autocast, time-based schedule.

Launch (Kaggle cell):  !torchrun --nproc_per_node=2 train.py --data_dir ... --out_dir /kaggle/working/run1 --budget_hours 6
Resume (same command + --resume auto) - total_steps / LR schedule / data position are restored from the checkpoint.
Data: random windows of (ctx+1) tokens drawn from the per-source uint16 .bin files with mixture weights.
The window for (seed, step, micro-step, rank) is deterministic, so resume continues the exact data stream.
"""
import argparse, glob, json, math, os, sys, time
from dataclasses import asdict

import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from legal_lm import LMConfig, LegalLM

p = argparse.ArgumentParser()
p.add_argument("--data_dir", default=None, help="folder with <src>_train.bin / <src>_val.bin (auto-found in /kaggle/input)")
p.add_argument("--out_dir", default="/kaggle/working/run1")
p.add_argument("--budget_hours", type=float, default=6.0, help="training time budget (excluding nothing: wall clock of training)")
p.add_argument("--mix", default="supreme_court:0.40,high_courts:0.25,tribunals:0.25,acts:0.10")
p.add_argument("--ctx", type=int, default=1024)
p.add_argument("--micro_bs", type=int, default=8, help="sequences per GPU per micro-step")
p.add_argument("--tokens_per_update", type=int, default=131072)
p.add_argument("--lr", type=float, default=6e-4)
p.add_argument("--min_lr_frac", type=float, default=0.1)
p.add_argument("--warmup_frac", type=float, default=0.03)
p.add_argument("--wd", type=float, default=0.1)
p.add_argument("--clip", type=float, default=1.0)
p.add_argument("--n_layer", type=int, default=12)
p.add_argument("--n_head", type=int, default=12)
p.add_argument("--d_model", type=int, default=768)
p.add_argument("--d_ff", type=int, default=2048)
p.add_argument("--ckpt_minutes", type=float, default=25)
p.add_argument("--keep_ckpts", type=int, default=3)
p.add_argument("--eval_every", type=int, default=500)
p.add_argument("--val_windows", type=int, default=64, help="fixed validation windows per source")
p.add_argument("--log_every", type=int, default=20)
p.add_argument("--seed", type=int, default=1234)
p.add_argument("--resume", default="", help="'auto' (latest in out_dir) or a checkpoint path")
p.add_argument("--max_steps", type=int, default=0, help="debug: override total steps")
p.add_argument("--smoke", action="store_true", help="tiny model/CPU-friendly debug run")
args = p.parse_args()

# ----------------------------------------------------------------------------- distributed setup
ddp = "RANK" in os.environ
rank = int(os.environ.get("RANK", 0)); world = int(os.environ.get("WORLD_SIZE", 1))
local_rank = int(os.environ.get("LOCAL_RANK", 0))
use_cuda = torch.cuda.is_available() and not args.smoke
if ddp:
    dist.init_process_group("nccl" if use_cuda else "gloo")
dev = torch.device(f"cuda:{local_rank}" if use_cuda else "cpu")
if use_cuda:
    torch.cuda.set_device(dev)
is_main = rank == 0
def log(*a):
    if is_main:
        print(f"[{time.strftime('%H:%M:%S')}]", *a, flush=True)

# ----------------------------------------------------------------------------- data
if args.data_dir is None:
    hits = glob.glob("/kaggle/input/**/supreme_court_train.bin", recursive=True)
    assert hits, "could not find *_train.bin under /kaggle/input - pass --data_dir"
    args.data_dir = os.path.dirname(hits[0])
mix = {k: float(v) for k, v in (kv.split(":") for kv in args.mix.split(","))}
srcs = list(mix); w = np.array([mix[s] for s in srcs]); w = w / w.sum()
train_bins = {s: np.memmap(f"{args.data_dir}/{s}_train.bin", dtype=np.uint16, mode="r") for s in srcs}
val_bins = {s: np.memmap(f"{args.data_dir}/{s}_val.bin", dtype=np.uint16, mode="r") for s in srcs}
W = args.ctx + 1

def get_batch(step, micro):
    rng = np.random.default_rng([args.seed, step, micro, rank])
    out = np.empty((args.micro_bs, W), dtype=np.int64)
    for i in range(args.micro_bs):
        s = srcs[rng.choice(len(srcs), p=w)]
        m = train_bins[s]
        o = int(rng.integers(0, len(m) - W))
        out[i] = m[o:o + W]
    t = torch.from_numpy(out).to(dev, non_blocking=True)
    return t[:, :-1].contiguous(), t[:, 1:].contiguous()

val_sets = {}
for s in srcs:
    r = np.random.default_rng(777)
    offs = r.integers(0, len(val_bins[s]) - W, size=args.val_windows)
    val_sets[s] = np.stack([val_bins[s][o:o + W] for o in offs]).astype(np.int64)

# ----------------------------------------------------------------------------- model
if args.smoke:
    args.n_layer, args.n_head, args.d_model, args.d_ff, args.ctx = 2, 4, 64, 128, 64
    W = args.ctx + 1
cfg = LMConfig(vocab_size=32000, n_layer=args.n_layer, n_head=args.n_head, d_model=args.d_model,
               d_ff=args.d_ff, max_seq_len=args.ctx)
torch.manual_seed(args.seed)                      # same init on every rank (DDP also broadcasts)
model = LegalLM(cfg).to(dev)
n_params = model.num_params()
log(f"parameters: {n_params:,} ({n_params/1e6:.2f}M)  config: {asdict(cfg)}")
assert 50e6 <= n_params <= 300e6 or args.smoke, "parameter count outside the 50M-300M rule"

decay = [q for n, q in model.named_parameters() if q.ndim >= 2 and not n.startswith("tok_emb")]
no_decay = [q for n, q in model.named_parameters() if q.ndim < 2 or n.startswith("tok_emb")]
opt = torch.optim.AdamW([{"params": decay, "weight_decay": args.wd}, {"params": no_decay, "weight_decay": 0.0}],
                        lr=args.lr, betas=(0.9, 0.95), eps=1e-8, fused=use_cuda)
scaler = torch.amp.GradScaler("cuda", enabled=use_cuda)
amp = torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_cuda)

accum = max(1, round(args.tokens_per_update / (args.micro_bs * args.ctx * world)))
tok_per_update = accum * args.micro_bs * args.ctx * world
log(f"world={world} micro_bs={args.micro_bs} accum={accum} -> {tok_per_update:,} tokens/update")

os.makedirs(args.out_dir, exist_ok=True)
ck_dir = os.path.join(args.out_dir, "ckpt"); os.makedirs(ck_dir, exist_ok=True)

# ----------------------------------------------------------------------------- resume
step, train_seconds, total_steps = 0, 0.0, None
resume = args.resume
if resume == "auto":
    cks = sorted(glob.glob(os.path.join(ck_dir, "ckpt_*.pt")))
    resume = cks[-1] if cks else ""
if resume:
    ck = torch.load(resume, map_location="cpu")
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    if use_cuda and "scaler" in ck:
        scaler.load_state_dict(ck["scaler"])
    step, train_seconds, total_steps = ck["step"], ck["train_seconds"], ck["total_steps"]
    log(f"RESUMED from {resume}: step {step}/{total_steps}, {train_seconds/3600:.2f} h already trained")
    del ck
raw = model
if ddp:
    model = DDP(model, device_ids=[local_rank] if use_cuda else None)

# ----------------------------------------------------------------------------- helpers
def micro_step(x, y, sync):
    ctx = model.no_sync() if (ddp and not sync) else torch.enable_grad()
    with ctx:
        with amp:
            _, loss = model(x, y)
        scaler.scale(loss / accum).backward()
    return loss.detach()

@torch.no_grad()
def evaluate():
    model.eval(); res = {}
    for s in srcs:
        arr = val_sets[s]; mine = arr[rank::world]
        tot = torch.zeros(2, device=dev)
        for i in range(0, len(mine), args.micro_bs):
            b = torch.from_numpy(mine[i:i + args.micro_bs]).to(dev)
            with amp:
                _, l = model(b[:, :-1].contiguous(), b[:, 1:].contiguous())
            tot += torch.tensor([l.item() * len(b), len(b)], device=dev)
        if ddp: dist.all_reduce(tot)
        res[s] = (tot[0] / tot[1]).item()
    model.train()
    res["mean"] = float(np.mean([res[s] for s in srcs]))
    return res

def lr_at(it):
    wu = max(20, int(args.warmup_frac * total_steps))
    if it < wu: return args.lr * (it + 1) / wu
    pr = (it - wu) / max(1, total_steps - wu)
    return args.lr * (args.min_lr_frac + (1 - args.min_lr_frac) * 0.5 * (1 + math.cos(math.pi * min(1.0, pr))))

def bcast_flag(v):
    t = torch.tensor([1.0 if v else 0.0], device=dev)
    if ddp: dist.broadcast(t, 0)
    return t.item() > 0.5

def save_ckpt(final=False):
    if not is_main: return
    path = os.path.join(ck_dir, f"ckpt_{step:07d}.pt")
    torch.save({"model": raw.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                "step": step, "train_seconds": train_seconds, "total_steps": total_steps,
                "config": asdict(cfg)}, path + ".tmp")
    os.replace(path + ".tmp", path)
    for old in sorted(glob.glob(os.path.join(ck_dir, "ckpt_*.pt")))[:-args.keep_ckpts]:
        os.remove(old)
    log(f"checkpoint saved: {path}")

# ----------------------------------------------------------------------------- init-loss check + throughput benchmark
model.train()
if total_steps is None:
    x, y = get_batch(0, 0)
    with torch.no_grad(), amp:
        _, l0 = model(x, y)
    log(f"initial loss {l0.item():.3f}  (expected ~ln(32000) = {math.log(32000):.3f})")
    assert abs(l0.item() - math.log(32000)) < 0.6 or args.smoke, "init loss far from ln(V): check init/tokenizer"
    n_warm, n_meas = 2, 3
    for it in range(n_warm + n_meas):
        if it == n_warm:
            if use_cuda: torch.cuda.synchronize()
            tb = time.time()
        for m_ in range(accum):
            micro_step(*get_batch(0, m_), sync=(m_ == accum - 1))
        opt.zero_grad(set_to_none=True)
    if use_cuda: torch.cuda.synchronize()
    sec_per_update = (time.time() - tb) / n_meas
    t = torch.tensor([sec_per_update], device=dev)
    if ddp: dist.broadcast(t, 0)
    sec_per_update = t.item()
    scaler = torch.amp.GradScaler("cuda", enabled=use_cuda)   # discard scale state from benchmark
    tok_s = tok_per_update / sec_per_update
    overhead = 0.96                                            # eval + checkpoint + logging
    total_steps = args.max_steps or int(args.budget_hours * 3600 * overhead / sec_per_update)
    log(f"benchmark: {sec_per_update:.2f} s/update = {tok_s:,.0f} tok/s | budget {args.budget_hours} h "
        f"-> total_steps {total_steps:,} = {total_steps*tok_per_update/1e6:,.0f}M tokens")
    for s_, wt in zip(srcs, w):
        log(f"   {s_:14s} weight {wt:.2f} -> {wt*total_steps*tok_per_update/1e6:6.0f}M tokens = "
            f"{wt*total_steps*tok_per_update/len(train_bins[s_]):.2f} epochs of its pool")
    json.dump({"args": vars(args), "params": n_params, "tok_per_s": tok_s, "total_steps": total_steps,
               "tokens_per_update": tok_per_update}, open(os.path.join(args.out_dir, "run_info.json"), "w"), indent=1)

# ----------------------------------------------------------------------------- training loop
hard_limit = args.budget_hours * 3600 * 1.04
logf = open(os.path.join(args.out_dir, "train_log.jsonl"), "a") if is_main else None
t_sess = time.time(); t_ck = time.time(); t_log = time.time(); loss_acc = 0.0; n_acc = 0
base_seconds = train_seconds
ev = evaluate()                                  # all ranks take part (all_reduce inside)
log("val @ step", step, {k: round(v, 3) for k, v in ev.items()})
while step < total_steps:
    for g in opt.param_groups: g["lr"] = lr_at(step)
    for m_ in range(accum):
        l = micro_step(*get_batch(step, m_), sync=(m_ == accum - 1))
        loss_acc += l.item() / accum
    scaler.unscale_(opt)
    gn = torch.nn.utils.clip_grad_norm_(raw.parameters(), args.clip)
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    step += 1; n_acc += 1
    train_seconds = base_seconds + (time.time() - t_sess)
    if step % args.log_every == 0:
        dt = time.time() - t_log; t_log = time.time()
        tps = args.log_every * tok_per_update / dt
        mem = torch.cuda.max_memory_allocated(dev) / 2**30 if use_cuda else 0
        eta = (total_steps - step) * dt / args.log_every / 3600
        rec = dict(step=step, loss=round(loss_acc / n_acc, 4), ppl=round(math.exp(loss_acc / n_acc), 2),
                   lr=round(lr_at(step), 7), gnorm=round(float(gn), 3), tok_s=round(tps), mem_gb=round(mem, 2),
                   hours=round(train_seconds / 3600, 3), eta_h=round(eta, 2), scale=scaler.get_scale() if use_cuda else 1)
        log(json.dumps(rec))
        if is_main: logf.write(json.dumps(rec) + "\n"); logf.flush()
        loss_acc = 0.0; n_acc = 0
    if step % args.eval_every == 0 or step == total_steps:
        ev = evaluate()
        log("VAL step", step, {k: f"{v:.3f} (ppl {math.exp(v):.1f})" for k, v in ev.items()})
        if is_main: logf.write(json.dumps({"step": step, "val": ev}) + "\n"); logf.flush()
    stop = bcast_flag(train_seconds > hard_limit)
    if bcast_flag((time.time() - t_ck) / 60 > args.ckpt_minutes) or stop:
        save_ckpt(); t_ck = time.time()
    if stop:
        log(f"TIME LIMIT reached at step {step}/{total_steps} - stopping with a saved checkpoint")
        break

# ----------------------------------------------------------------------------- final save
save_ckpt()
if is_main:
    fin = os.path.join(args.out_dir, "final"); os.makedirs(fin, exist_ok=True)
    raw.save_pretrained(os.path.join(fin, "model.pt"))                  # fp32 + model_config.json
    raw.save_pretrained(os.path.join(fin, "model_fp16.pt"), half=True)  # small copy for inference
    log(f"DONE: step {step}, {train_seconds/3600:.2f} h. Final weights in {fin}")
if ddp:
    dist.barrier(); dist.destroy_process_group()

Writing /kaggle/working/train.py


In [5]:
%%writefile /kaggle/working/prep_data.py
"""prep_data.py - stream the 4 official JSON files, normalize, tokenize (new tokenizer), write uint16 .bin per source.
Random spread over each whole file (hash-based Bernoulli keep), document-level deterministic val split,
</s> between documents. Usage: python prep_data.py --out DIR [--smoke]
Env: HF_TOKEN (HuggingFace token). Re-running skips sources whose <src>_meta.json exists.
"""
import argparse, glob, hashlib, json, os, sys, time
from collections import deque
from multiprocessing import Pool

import numpy as np, requests, ijson

ap = argparse.ArgumentParser()
ap.add_argument("--out", required=True)
ap.add_argument("--smoke", action="store_true")
ap.add_argument("--max_min_per_source", type=float, default=35, help="safety cap on streaming time per source")
args = ap.parse_args()

sys.path.insert(0, os.path.dirname(glob.glob("/kaggle/input/**/a1_tokenization.py", recursive=True)[0]))
from a1_tokenization import Tokenizer, normalize_text, EOS_ID, CountingReader
tok = Tokenizer.load(os.path.dirname(glob.glob("/kaggle/input/**/vocab.json", recursive=True)[0]))
assert tok.decode(tok.encode("Test  \n judgment.")) == "Test judgment."

print("ijson backend:", ijson.backend, "(yajl2_c = fast; 'python' = slow)", flush=True)
OUT = args.out; os.makedirs(OUT, exist_ok=True)
NUM_WORKERS, MIN_WORDS, VAL_FRAC, OVERSAMPLE, TEXT_FRAC = 3, 30, 0.01, 1.3, 0.9
ORDER = ["supreme_court", "acts", "tribunals", "high_courts"]
TARGETS = {"supreme_court": 300e6, "high_courts": 300e6, "tribunals": 300e6, "acts": 80e6}   # train tokens (pool)
FILES = {"acts": "train_acts.json", "high_courts": "train_high_courts.json",
         "supreme_court": "train_supreme_court.json", "tribunals": "train_tribunals.json"}
SIZES = {"acts": 0.729e9, "high_courts": 10.7e9, "supreme_court": 1.32e9, "tribunals": 6.48e9}
EVAL_PER_SRC = {"supreme_court": 150, "high_courts": 150, "tribunals": 150, "acts": 40}
EVAL_MIN_WORDS, BATCH_CHARS = 700, 1_000_000
BASE = "https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Train/"
HF_TOKEN = os.environ.get("HF_TOKEN", "")


def work(a):
    is_val, texts = a
    ids, n_docs, n_words, ev = [], 0, 0, []
    for raw in texts:
        t = normalize_text(raw)                      # same normalization as the test prompts (no newlines)
        nw = t.count(" ") + 1
        if nw < MIN_WORDS: continue
        ids.extend(tok.encode(t, normalize=False)); ids.append(EOS_ID)
        n_docs += 1; n_words += nw
        if is_val and nw >= EVAL_MIN_WORDS: ev.append(t)
    arr = np.asarray(ids, dtype=np.uint16)
    return is_val, arr.tobytes(), n_docs, n_words, len(arr), ev


def hash_u(text):
    d = hashlib.blake2b(text[:400].encode("utf-8", "ignore"), digest_size=16).digest()
    return int.from_bytes(d[:8], "big") / 2**64, int.from_bytes(d[8:], "big") / 2**64


def run_pass(name, pool, target, p_keep):
    r = requests.get(BASE + FILES[name], headers={"Authorization": f"Bearer {HF_TOKEN}"}, stream=True, timeout=(30, 300))
    r.raise_for_status(); r.raw.decode_content = True
    rd = CountingReader(r.raw)
    ftr = open(f"{OUT}/{name}_train.bin", "wb"); fva = open(f"{OUT}/{name}_val.bin", "wb")
    st = dict(train_tok=0, val_tok=0, train_docs=0, val_docs=0, words=0, seen=0)
    eval_texts, pending = [], deque()
    buf, chars = {False: [], True: []}, {False: 0, True: 0}

    def collect():
        is_val, b, nd, nw, nt, ev = pending.popleft().get()
        (fva if is_val else ftr).write(b)
        k = "val" if is_val else "train"
        st[k + "_tok"] += nt; st[k + "_docs"] += nd; st["words"] += nw
        for t in ev:
            if len(eval_texts) < EVAL_PER_SRC[name]: eval_texts.append(t)

    def submit(is_val):
        if not buf[is_val]: return
        pending.append(pool.apply_async(work, ((is_val, buf[is_val]),)))
        buf[is_val], chars[is_val] = [], 0
        while len(pending) > 2 * NUM_WORKERS + 2: collect()

    t0 = last = time.time(); stopped = "end_of_file"
    try: it = ijson.items(rd, "item", use_float=True)
    except TypeError: it = ijson.items(rd, "item")
    for doc in it:
        st["seen"] += 1
        text = doc.get("text") if isinstance(doc, dict) else None
        if not text or len(text) < 200: continue
        u1, u2 = hash_u(text)
        if u1 >= p_keep: continue
        is_val = u2 < VAL_FRAC
        buf[is_val].append(text); chars[is_val] += len(text)
        if chars[is_val] >= BATCH_CHARS or len(buf[is_val]) >= 256: submit(is_val)
        if st["train_tok"] >= target: stopped = "target_reached"; break
        if time.time() - t0 > args.max_min_per_source * 60: stopped = "time_cap"; break
        if time.time() - last > 30:
            last = time.time()
            print(f"[{name}] seen {st['seen']:,} | {rd.n/1e6:,.0f} MB ({rd.n/SIZES[name]:.0%}) @ {rd.n/1e6/(last-t0):.0f} MB/s | "
                  f"train {st['train_tok']/1e6:,.1f}M tok | val {st['val_tok']/1e6:.2f}M", flush=True)
    r.close()
    submit(False); submit(True)
    while pending: collect()
    ftr.close(); fva.close()
    with open(f"{OUT}/{name}_eval.jsonl", "w") as f:
        for t in eval_texts: f.write(json.dumps({"src": name, "text": t}) + "\n")
    meta = dict(source=name, p_keep=p_keep, target=target, stopped=stopped, file_fraction_read=round(rd.n / SIZES[name], 3),
                eval_docs=len(eval_texts), tokens_per_word=round((st["train_tok"] + st["val_tok"]) / max(st["words"], 1), 4),
                minutes=round((time.time() - t0) / 60, 1), **st)
    json.dump(meta, open(f"{OUT}/{name}_meta.json", "w"), indent=1)   # written last = "done" marker
    return meta


def process_source(name, pool):
    if os.path.exists(f"{OUT}/{name}_meta.json"):
        print(f"[{name}] already done, skipping", flush=True); return
    est = SIZES[name] * TEXT_FRAC / 4.7
    target = 3e6 if args.smoke else TARGETS[name]
    p_keep = 1.0 if args.smoke else min(1.0, OVERSAMPLE * target / est)
    print(f"[{name}] target {target/1e6:.0f}M tok | est. file tokens {est/1e6:,.0f}M | p_keep {p_keep:.3f}", flush=True)
    for attempt in range(3):
        try:
            print(f"[{name}] DONE", run_pass(name, pool, target, p_keep), flush=True); return
        except Exception as e:
            print(f"[{name}] attempt {attempt+1} failed: {repr(e)[:200]}; retrying", flush=True); time.sleep(10)
    raise RuntimeError(f"{name} failed 3 times")


if __name__ == "__main__":
    assert HF_TOKEN, "HF_TOKEN env var missing"
    with Pool(NUM_WORKERS) as pool:
        for n in ORDER: process_source(n, pool)
    # ---- verification ----
    tot = 0
    for n in ORDER:
        for split in ("train", "val"):
            m = np.memmap(f"{OUT}/{n}_{split}.bin", dtype=np.uint16, mode="r")
            mx = max(int(m[i:i + 50_000_000].max()) for i in range(0, len(m), 50_000_000))
            assert mx < 32000, (n, split, mx)
            print(f"{n:14s} {split:5s} {len(m)/1e6:8.2f}M tokens | max id {mx}")
            if split == "train": tot += len(m)
    print(f"TOTAL train tokens: {tot/1e6:,.1f}M")
    m = np.memmap(f"{OUT}/supreme_court_train.bin", dtype=np.uint16, mode="r")
    print("sample:", tok.decode(m[:80].tolist()))

Writing /kaggle/working/prep_data.py


In [6]:
%%writefile /kaggle/working/test_model.py
import torch, time
from legal_lm import *
torch.manual_seed(0)
cfg = LMConfig(n_layer=2, n_head=4, d_model=64, d_ff=128, max_seq_len=128, vocab_size=500)
m = LegalLM(cfg).eval()
print("params", m.num_params())
x = torch.randint(4, 500, (1, 20))
_, loss = m(x[:, :-1], x[:, 1:]); print("init loss", loss.item(), "ln V", math.log(500))
# KV-cache correctness incl. left padding: greedy cached vs full-forward greedy
prompts = [torch.randint(4, 500, (n,)).tolist() for n in (12, 7, 20)]
sw = torch.zeros(500, dtype=torch.long)
out = generate_batch(m, prompts, 10, sw, target_words=10**6)
ok = True
for p, o in zip(prompts, out):
    seq = list(p)
    for _ in range(10):
        lg, _ = m(torch.tensor([seq])); l = lg[0, -1].clone(); l[:4] = -1e9
        seq.append(int(l.argmax()))
    ok &= seq[len(p):] == o
print("cache == full forward (batched, left-padded):", ok)

Writing /kaggle/working/test_model.py


## 1) Fail-early checks (model unit test, tiny data-prep smoke, tiny training smoke)

In [7]:
!cd /kaggle/working && python test_model.py

params 114240
init loss 6.199282169342041 ln V 6.214608098422191
cache == full forward (batched, left-padded): True


In [8]:
!cd /kaggle/working && python prep_data.py --out /kaggle/working/data_smoke --smoke

ijson backend: yajl2_c (yajl2_c = fast; 'python' = slow)
[supreme_court] target 3M tok | est. file tokens 253M | p_keep 1.000
[supreme_court] DONE {'source': 'supreme_court', 'p_keep': 1.0, 'target': 3000000.0, 'stopped': 'target_reached', 'file_fraction_read': 0.019, 'eval_docs': 5, 'tokens_per_word': 1.2241, 'minutes': 0.1, 'train_tok': 4839590, 'val_tok': 42139, 'train_docs': 704, 'val_docs': 5, 'words': 3988058, 'seen': 709}
[acts] target 3M tok | est. file tokens 140M | p_keep 1.000
[acts] DONE {'source': 'acts', 'p_keep': 1.0, 'target': 3000000.0, 'stopped': 'target_reached', 'file_fraction_read': 0.035, 'eval_docs': 2, 'tokens_per_word': 1.2555, 'minutes': 0.1, 'train_tok': 5243806, 'val_tok': 15991, 'train_docs': 580, 'val_docs': 3, 'words': 4189345, 'seen': 583}
[tribunals] target 3M tok | est. file tokens 1,241M | p_keep 1.000
[tribunals] DONE {'source': 'tribunals', 'p_keep': 1.0, 'target': 3000000.0, 'stopped': 'target_reached', 'file_fraction_read': 0.004, 'eval_docs': 13,

In [9]:
!cd /kaggle/working && python train.py --smoke --data_dir /kaggle/working/data_smoke --max_steps 6 --out_dir /kaggle/working/smoke --tokens_per_update 512 --micro_bs 2 --val_windows 4 --log_every 2 --eval_every 3

[05:08:16] parameters: 2,130,240 (2.13M)  config: {'vocab_size': 32000, 'n_layer': 2, 'n_head': 4, 'd_model': 64, 'd_ff': 128, 'max_seq_len': 64, 'rope_theta': 10000.0, 'norm_eps': 1e-05, 'dropout': 0.0, 'tie_embeddings': True, 'pad_id': 0, 'bos_id': 1, 'eos_id': 2}
[05:08:23] world=1 micro_bs=2 accum=4 -> 512 tokens/update
[05:08:23] initial loss 10.418  (expected ~ln(32000) = 10.373)
[05:08:24] benchmark: 0.19 s/update = 2,754 tok/s | budget 6.0 h -> total_steps 6 = 0M tokens
[05:08:24]    supreme_court  weight 0.40 ->      0M tokens = 0.00 epochs of its pool
[05:08:24]    high_courts    weight 0.25 ->      0M tokens = 0.00 epochs of its pool
[05:08:24]    tribunals      weight 0.25 ->      0M tokens = 0.00 epochs of its pool
[05:08:24]    acts           weight 0.10 ->      0M tokens = 0.00 epochs of its pool
[05:08:26] val @ step 0 {'supreme_court': 10.385, 'high_courts': 10.387, 'tribunals': 10.379, 'acts': 10.372, 'mean': 10.381}
[05:08:27] {"step": 2, "loss": 10.3811, "ppl": 3224

## 2) Real data prep (resumable: finished sources are skipped on rerun)

In [10]:
!cd /kaggle/working && python prep_data.py --out /kaggle/working/data
!rm -rf /kaggle/working/data_smoke /kaggle/working/smoke

ijson backend: yajl2_c (yajl2_c = fast; 'python' = slow)
[supreme_court] target 300M tok | est. file tokens 253M | p_keep 1.000
[supreme_court] seen 10,759 | 266 MB (20%) @ 9 MB/s | train 49.7M tok | val 0.21M
[supreme_court] seen 25,279 | 578 MB (44%) @ 10 MB/s | train 109.3M tok | val 0.86M
[supreme_court] seen 37,647 | 874 MB (66%) @ 10 MB/s | train 165.4M tok | val 1.49M
[supreme_court] seen 45,296 | 1,103 MB (84%) @ 9 MB/s | train 210.8M tok | val 1.92M
[supreme_court] DONE {'source': 'supreme_court', 'p_keep': 1.0, 'target': 300000000.0, 'stopped': 'end_of_file', 'file_fraction_read': 1.002, 'eval_docs': 150, 'tokens_per_word': 1.2258, 'minutes': 2.4, 'train_tok': 255291006, 'val_tok': 2368475, 'train_docs': 54915, 'val_docs': 560, 'words': 210194188, 'seen': 55549}
[acts] target 80M tok | est. file tokens 140M | p_keep 0.745
[acts] seen 8,179 | 327 MB (45%) @ 11 MB/s | train 49.9M tok | val 0.21M
[acts] DONE {'source': 'acts', 'p_keep': 0.7450083828684652, 'target': 80000000.0, 

## 3) Training (DDP on 2 GPUs). If the session dies: rerun all cells, training resumes from the last checkpoint (`--resume auto`).

In [11]:
!cd /kaggle/working && torchrun --nproc_per_node=2 train.py --data_dir /kaggle/working/data --out_dir /kaggle/working/run1 --budget_hours {BUDGET_HOURS} --mix {MIX} --resume auto

W1006 05:17:44.204000 102 torch/distributed/run.py:851] 
W1006 05:17:44.204000 102 torch/distributed/run.py:851] *****************************************
W1006 05:17:44.204000 102 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1006 05:17:44.204000 102 torch/distributed/run.py:851] *****************************************
[05:17:48] parameters: 109,529,856 (109.53M)  config: {'vocab_size': 32000, 'n_layer': 12, 'n_head': 12, 'd_model': 768, 'd_ff': 2048, 'max_seq_len': 1024, 'rope_theta': 10000.0, 'norm_eps': 1e-05, 'dropout': 0.0, 'tie_embeddings': True, 'pad_id': 0, 'bos_id': 1, 'eos_id': 2}
[05:17:49] world=2 micro_bs=8 accum=8 -> 131,072 tokens/update
[05:17:52] initial loss 10.494  (expected ~ln(32000) = 10.373)
[05:18:13] benchmark: 4.26 s/update = 30,787 tok/s | budget 7.5 h -> total_ste

## 4) Decoding sweep on held-out docs, then the 1000 test outputs

In [12]:
import sys, json, time, random, zipfile, numpy as np, torch
sys.path.insert(0, "/kaggle/working")
sys.path.insert(0, os.path.dirname(glob.glob("/kaggle/input/**/a1_tokenization.py", recursive=True)[0]))
from a1_tokenization import Tokenizer
from legal_lm import load_legal_lm, generate_texts
tok = Tokenizer.load(os.path.dirname(glob.glob("/kaggle/input/**/vocab.json", recursive=True)[0]))
dev = "cuda"
model = load_legal_lm("/kaggle/working/run1/final/model_fp16.pt", device=dev, dtype=torch.float16)
print("params:", f"{model.num_params():,}")
random.seed(0)
docs = [json.loads(l) for f in sorted(glob.glob("/kaggle/working/data/*_eval.jsonl")) for l in open(f)]
pairs = []
for d in docs:
    ws = d["text"].split(); s = random.randint(0, len(ws) - 640)
    pairs.append((" ".join(ws[s:s+512]), " ".join(ws[s+512:s+640])))
random.shuffle(pairs); pairs = pairs[:150]
P = [p[0] for p in pairs]; R = [p[1] for p in pairs]
print("held-out pairs:", len(pairs))

params: 109,529,856
held-out pairs: 150


In [13]:
from bert_score import score as bert_score
from rouge_score import rouge_scorer
rs = rouge_scorer.RougeScorer(["rougeL"])
SWEEP = {
 "greedy":             dict(temperature=0.0),
 "greedy_rep1.1":      dict(temperature=0.0, rep_penalty=1.1),
 "greedy_ng4":         dict(temperature=0.0, no_repeat_ngram=4),
 "greedy_rep1.1_ng4":  dict(temperature=0.0, rep_penalty=1.1, no_repeat_ngram=4),
 "greedy_rep1.05_ng3": dict(temperature=0.0, rep_penalty=1.05, no_repeat_ngram=3),
 "t0.3_p0.9_ng4":      dict(temperature=0.3, top_p=0.9, no_repeat_ngram=4),
}
results, BEST = {}, "greedy_rep1.1_ng4"          # BEST = fallback if the sweep fails
try:
    for name, cfg in SWEEP.items():
        t0 = time.time()
        out = generate_texts(model, tok, P, gen_cfg=cfg, batch_size=40, verbose=False)
        _, _, f = bert_score(out, R, model_type="roberta-large", lang="en", batch_size=32, device=dev)
        rl = float(np.mean([rs.score(r, o)["rougeL"].fmeasure for r, o in zip(R, out)]))
        results[name] = dict(bertscore=float(f.mean()), rougeL=rl)
        print(f"{name:20s} BERTScore {results[name]['bertscore']:.4f} | ROUGE-L {rl:.4f} | {time.time()-t0:.0f}s")
    BEST = max(results, key=lambda k: results[k]["bertscore"])
except Exception as e:
    print("SWEEP FAILED, using fallback setting:", repr(e)[:300])
json.dump(results, open("/kaggle/working/decoding_sweep.json", "w"), indent=1)
print("BEST:", BEST)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

... (15 outputs left out)

greedy_rep1.05_ng3   BERTScore 0.8387 | ROUGE-L 0.2395 | 46s


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


t0.3_p0.9_ng4        BERTScore 0.8401 | ROUGE-L 0.2501 | 48s
BEST: t0.3_p0.9_ng4


In [14]:
from huggingface_hub import hf_hub_download
TEST = hf_hub_download("Exploration-Lab/CS-781-Capstone", "Phase_1_Pretraining/Test/test.json", repo_type="dataset", token=os.environ["HF_TOKEN"])
test = json.load(open(TEST)); ids = sorted(test, key=int)
print("test prompts:", len(ids), "| decoding:", BEST, SWEEP[BEST])
texts = generate_texts(model, tok, [test[i]["text"] for i in ids], gen_cfg=SWEEP[BEST], batch_size=40)
pred = {i: {"text": t} for i, t in zip(ids, texts)}
assert set(pred) == set(test) and len(pred) == 1000
bad = [i for i, v in pred.items() if len(v["text"].split()) != 128 or not v["text"].strip()]
assert not bad, bad[:10]
assert not any(pred[i]["text"].startswith(test[i]["text"][:60]) for i in ids), "prompt leaked into output"
os.makedirs("/kaggle/working/sub", exist_ok=True)
json.dump(pred, open("/kaggle/working/sub/prediction.json", "w"), ensure_ascii=False)
ZIP = "/kaggle/working/submission.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z: z.write("/kaggle/working/sub/prediction.json", arcname="prediction.json")
with zipfile.ZipFile(ZIP) as z:
    assert z.namelist() == ["prediction.json"]; back = json.loads(z.read("prediction.json"))
assert set(back) == set(test) and all(len(v["text"].split()) == 128 for v in back.values())
print("OK:", ZIP, f"{os.path.getsize(ZIP)/1e6:.2f} MB | 1000 ids | 128 words each | prediction.json at zip root")
print("Sample:", back["1"]["text"][:400])

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

test prompts: 1000 | decoding: t0.3_p0.9_ng4 {'temperature': 0.3, 'top_p': 0.9, 'no_repeat_ngram': 4}
  round 2 batch 1/1 (135s)
  generated 1000 texts in 135s; fallback used for 1
OK: /kaggle/working/submission.zip 0.20 MB | 1000 ids | 128 words each | prediction.json at zip root
Sample: well settled that a writ petition under Article 226 of the Constitution is not maintainable when the same relief sought at the time of seeking the same relief had been denied. The High Court has jurisdiction to quash the notification under Section 4 of the Act. The High court has no jurisdiction to entertain a writ petition. The High Courts have jurisdiction to entertain the writ petition. It is w
